In [ ]:
!nvidia-smi
!pip install torch numpy pandas scikit-learn matplotlib


Wed Nov 19 16:50:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df_train = pd.read_csv("/content/drive/My Drive/df_daily_final.csv")
df_test = pd.read_csv("/content/drive/My Drive/df_daily_eval_final.csv")

df_train['dt'] = pd.to_datetime(df_train['dt'])
df_test['dt'] = pd.to_datetime(df_test['dt'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
target = "sale_amount_log"

categorical_cols = [
    'store_id', 'management_group_id', 'first_category_id',
    'second_category_id', 'third_category_id', 'product_id',
    'holiday_flag', 'activity_flag',
    'binary_stockout','is_month_start','is_month_end'
]

drop_cols = ['dt', 'sale_amount_log']

# encode categoricals
for col in categorical_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col])
    df_test[col] = le.transform(df_test[col])


In [ ]:
columns_to_drop = [
    'day_of_year_sin','day_of_year_cos','quarter_cos',
    'rolling_std_7_log','rolling_std_14_log','lag_1_discount',
    'prev_day_hourly_stock_active_ratio_active','week_of_year_sin',
    'week_of_year_cos','quarter_sin','days_since_last_promo',
    'stock_recovery','is_weekend','cv_14_log','roll_mean_28_log',
    'month_cos','month_sin'
]

df_train = df_train.drop(columns=columns_to_drop, errors='ignore')
df_test = df_test.drop(columns=columns_to_drop, errors='ignore')


In [ ]:
def create_sequences(df, seq_len=30):
    X, y = [], []
    features = df.drop(columns=['dt', target]).values
    target_vals = df[target].values

    for i in range(len(df) - seq_len):
        X.append(features[i:i+seq_len])
        y.append(target_vals[i+seq_len])
    return np.array(X), np.array(y)

SEQ_LEN = 30
X_train, y_train = create_sequences(df_train, SEQ_LEN)
X_test, y_test = create_sequences(df_test, SEQ_LEN)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).to(device)


In [ ]:
class AttentionLSTM(nn.Module):
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.attn = nn.Linear(hidden, 1)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = (weights * lstm_out).sum(dim=1)
        return self.fc(context)

model = AttentionLSTM(X_train.shape[2]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
EPOCHS = 30
BATCH = 256

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(X_train_t))
    Xb = X_train_t[perm]
    yb = y_train_t[perm]

    for i in range(0, len(Xb), BATCH):
        xb = Xb[i:i+BATCH]
        yb_ = yb[i:i+BATCH]

        optimizer.zero_grad()
        pred = model(xb).squeeze()
        loss = criterion(pred, yb_)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss.item():.4f}")


Epoch 1/30 - Loss: 0.0658
Epoch 2/30 - Loss: 0.0523
Epoch 3/30 - Loss: 0.0593
Epoch 4/30 - Loss: 0.0506
Epoch 5/30 - Loss: 0.0527
Epoch 6/30 - Loss: 0.0599
Epoch 7/30 - Loss: 0.0461
Epoch 8/30 - Loss: 0.0477
Epoch 9/30 - Loss: 0.0591
Epoch 10/30 - Loss: 0.0433
Epoch 11/30 - Loss: 0.0455
Epoch 12/30 - Loss: 0.0535
Epoch 13/30 - Loss: 0.0537
Epoch 14/30 - Loss: 0.0413
Epoch 15/30 - Loss: 0.0484
Epoch 16/30 - Loss: 0.0598
Epoch 17/30 - Loss: 0.0516
Epoch 18/30 - Loss: 0.0595
Epoch 19/30 - Loss: 0.0498
Epoch 20/30 - Loss: 0.0428
Epoch 21/30 - Loss: 0.0469
Epoch 22/30 - Loss: 0.0559
Epoch 23/30 - Loss: 0.0515
Epoch 24/30 - Loss: 0.0389
Epoch 25/30 - Loss: 0.0565
Epoch 26/30 - Loss: 0.0630
Epoch 27/30 - Loss: 0.0535
Epoch 28/30 - Loss: 0.0480
Epoch 29/30 - Loss: 0.0471
Epoch 30/30 - Loss: 0.0603


In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test_t).squeeze()

y_test_orig = np.expm1(y_test)
preds_orig = np.expm1(preds.cpu().numpy())

rmse = np.sqrt(mean_squared_error(y_test_orig, preds_orig))
r2 = r2_score(y_test_orig, preds_orig)

print("RMSE:", rmse)
print("R2:", r2)


RMSE: 1.4311432139800948
R2: 0.20479734312174303


In [ ]:
df_train.columns

Index(['store_id', 'management_group_id', 'first_category_id',
       'second_category_id', 'third_category_id', 'product_id', 'dt',
       'stock_hour6_22_cnt', 'discount', 'holiday_flag', 'activity_flag',
       'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level',
       'sale_amount_log', 'lag_1_log', 'lag_7_log', 'lag_14_log',
       'roll_mean_7_log', 'roll_mean_14_log', 'is_month_start', 'is_month_end',
       'day_of_week_sin', 'day_of_week_cos', 'cv_7_log', 'binary_stockout',
       'sale_stock_ratio_previous_day', 'prev_sale_stock_interaction',
       'lag_1_roll_14_interaction', 'prev_day_hourly_sales_mean_active',
       'prev_day_hourly_stockout_ratio_active', 'days_since_last_stockout'],
      dtype='object')